# Entraînement LoRA Stable Diffusion 3.5 Medium — personnage Solle

Notebook **Kaggle** (GPU **T4 single, 16 Go** suffit) pour entraîner un LoRA **SD 3.5 Medium** sur Solle.

**Pourquoi SD 3.5 Medium (et pas FLUX) :** 2,5B params (vs 12B FLUX) → tient sur **un seul T4**,
**sans** le `split_model_over_gpus` fragile qui faisait OOM. Architecture moderne (encodeur **T5**) =
gros bond de respect du prompt vs SDXL. Tes captions langage naturel sont parfaites pour lui.

**Clé anti-OOM :** on met le T5 en cache (`cache_text_embeddings`) puis on le **décharge**
(`unload_text_encoder`) → il ne reste que le transformer 2,5B quantisé en VRAM pendant le training.

> ⚠️ SD 3.5 Medium est **gated** sur HF : token OBLIGATOIRE.
> 1) Accepte la licence : https://huggingface.co/stabilityai/stable-diffusion-3.5-medium
> 2) Token Read : https://huggingface.co/settings/tokens
> 3) Ajoute-le en secret Kaggle nommé `HF_TOKEN` (Add-ons → Secrets), coché pour ce notebook.

Licence SD 3.5 = Stability Community : **usage commercial OK** sous seuil de CA (large pour une mascotte).

## 1. Vérifier le GPU

Un seul **T4** suffit. Inutile de prendre T4×2 ici.

In [ ]:
!nvidia-smi

## 2. Installer ai-toolkit

Même outil que pour FLUX ; il supporte SD 3.5.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/ostris/ai-toolkit
%cd ai-toolkit
!git submodule update --init --recursive
!pip install -q -r requirements.txt
!pip install -q bitsandbytes>=0.43.0
print('Installation terminée.')

## 3. Connexion HuggingFace — OBLIGATOIRE (SD 3.5 est gated)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN manquant ! SD 3.5 Medium est gated.\n"
        "1) Accepte la licence : https://huggingface.co/stabilityai/stable-diffusion-3.5-medium\n"
        "2) Cree un token Read : https://huggingface.co/settings/tokens\n"
        "3) Add-ons -> Secrets -> nom 'HF_TOKEN', coche ce notebook."
    )
login(token=hf_token)
print('Connecté à HuggingFace.')

## 4. Préparer le dataset

ai-toolkit attend un **dossier plat** : images + `.txt` côte à côte (même nom de base).
Ajoute ton dataset `solle-dataset-v3` via **Add Input** (panneau de droite).

In [ ]:
import os, glob, shutil

INPUT_ROOT = '/kaggle/input'
DST = '/kaggle/working/train_data'
os.makedirs(DST, exist_ok=True)

print('Datasets montés sous /kaggle/input/ :')
for d in sorted(glob.glob(f'{INPUT_ROOT}/*')):
    print('  -', d)

found = []
for ext in ('png', 'jpg', 'jpeg', 'txt'):
    found += glob.glob(f'{INPUT_ROOT}/**/*.{ext}', recursive=True)
found = sorted(set(found))
for f in found:
    shutil.copy(f, DST)

imgs = len(glob.glob(DST + '/*.png')) + len(glob.glob(DST + '/*.jpg')) + len(glob.glob(DST + '/*.jpeg'))
txts = len(glob.glob(DST + '/*.txt'))
print(f'\nFichiers copiés : images={imgs} | captions={txts}')
assert imgs > 0, 'AUCUNE image ! Ajoute le dataset via Add Input.'
assert imgs == txts, f'Mismatch ! {imgs} images vs {txts} captions'
print('Exemple caption :', open(sorted(glob.glob(DST + '/*.txt'))[0]).read())

## 5. Configuration d'entraînement (SD 3.5 Medium)

Différences vs FLUX :
- `name_or_path: stable-diffusion-3.5-medium` + **`is_v3: true`** (au lieu de `is_flux`)
- **PAS de `split_model_over_gpus`** : un seul T4 suffit pour 2,5B
- `quantize` + `low_vram` + `cache_text_embeddings` + `unload_text_encoder` → tient en 16 Go
- `sample_steps: 28` + `guidance_scale: 4.5` (SD 3.5 = ~25-30 steps, pas 4 comme Schnell)
- `steps: 2000`, checkpoints toutes les 500 → on choisit le meilleur
- `resolution: [1024]` → si OOM, repasse à `[768]`

In [ ]:
import os, yaml

config = {
    'job': 'extension',
    'config': {
        'name': 'solle_sd35',
        'process': [{
            'type': 'sd_trainer',
            'training_folder': '/kaggle/working/output',
            'performance_log_every': 100,
            'device': 'cuda:0',

            'network': {'type': 'lora', 'linear': 16, 'linear_alpha': 16},

            'save': {'dtype': 'float16', 'save_every': 500, 'max_step_saves_to_keep': 4},

            'datasets': [{
                'folder_path': '/kaggle/working/train_data',
                'caption_ext': 'txt',
                'caption_dropout_rate': 0.0,
                'shuffle_tokens': False,
                'cache_latents_to_disk': True,
                'cache_text_embeddings': True,
                'resolution': [1024],
            }],

            'train': {
                'batch_size': 1,
                'steps': 2000,
                'gradient_accumulation_steps': 1,
                'train_unet': True,
                'train_text_encoder': False,
                'unload_text_encoder': True,
                'gradient_checkpointing': True,
                'noise_scheduler': 'flowmatch',
                'optimizer': 'adamw8bit',
                'lr': 1e-4,
                'ema_config': {'use_ema': True, 'ema_decay': 0.99},
                'dtype': 'bf16',
            },

            'model': {
                'name_or_path': 'stabilityai/stable-diffusion-3.5-medium',
                'is_v3': True,
                'quantize': True,
                'low_vram': True,
            },

            'sample': {
                'sampler': 'flowmatch',
                'sample_every': 500,
                'width': 1024, 'height': 1024,
                'prompts': [
                    'sollechar, a purple furry monster with shaggy fur, mismatched yellow eyes, drawn in a comic illustration style, full body standing in a city park',
                    'sollechar, purple furry monster, sitting at a desk typing on a laptop in a cozy office',
                    'sollechar, purple furry monster, walking in a rainy city street wearing a yellow rain jacket and holding an umbrella',
                ],
                'neg': '',
                'seed': 42, 'walk_seed': True,
                'guidance_scale': 4.5,
                'sample_steps': 28,
            },
        }],
    },
    'meta': {'name': '[name]', 'version': '1.0'},
}

os.makedirs('/kaggle/working/configs', exist_ok=True)
cfg_path = '/kaggle/working/configs/solle_sd35.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
print('Config écrite :')
print(open(cfg_path).read())

## 6. Lancer l'entraînement

Une commande. Logs à surveiller :
- `loss` qui descend
- Samples auto toutes les 500 steps dans `output/.../samples/`
- Durée estimée : ~2-4 h sur T4.

In [ ]:
%cd /kaggle/working/ai-toolkit
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python run.py /kaggle/working/configs/solle_sd35.yaml

## 7. Récupérer le LoRA

Fichiers dans `/kaggle/working/output/`. Télécharge via l'onglet **Output** (droite).
Garde `solle_sd35.safetensors` (final) + un checkpoint intermédiaire (backup sur-entraînement).
Juge les samples pour choisir le meilleur step.

In [ ]:
import os, glob
output_dir = '/kaggle/working/output'
loras = sorted(glob.glob(f'{output_dir}/**/*.safetensors', recursive=True))
samples = sorted(glob.glob(f'{output_dir}/**/samples/*.png', recursive=True))
print(f'LoRA ({len(loras)}) :')
for f in loras:
    print(f'  {os.path.basename(f)} — {os.path.getsize(f)/1024/1024:.1f} Mo')
print(f'\nSamples ({len(samples)}), 6 derniers :')
for f in samples[-6:]:
    print(' ', f)